In [1]:
short_run = True

ai_only = False

import os
import sys
import csv
import json
import hashlib
from pathlib import Path

os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')

here = Path.cwd().resolve()

project = None

for folder in [here, *here.parents]:
    for path in [folder / 'app' / 'flatfairproj', folder / 'flatfairproj', folder]:
        if (path / 'manage.py').is_file():
            project = path

            break

    if project is not None:
        break

if project is None:
    raise Exception('Run this notebook from inside the FlatFair project folder.')

app_folder = project.parent

training = project / 'assets' / 'training'

photo_folder = training / 'photo_data'

clip_folder = project / 'assets' / 'ml' / 'clip'

saved_head_folder = project / 'assets' / 'ml'

saved_metrics_file = training / 'photo_metrics.json'

saved_embedding_file = training / 'photo_embeddings.npz'

if short_run:
    head_folder = training / 'check_runs' / 'photo'

    metrics_file = head_folder / 'photo_metrics.json'

    embedding_file = head_folder / 'photo_embeddings.npz'

    embedding_files = [saved_embedding_file, embedding_file]

    room_strengths = [0.1, 1]

    room_max_iter = 80

    mlp_max_iter = 40

    ai_max_iter = 80
else:
    head_folder = saved_head_folder

    metrics_file = saved_metrics_file

    embedding_file = saved_embedding_file

    embedding_files = [embedding_file]

    room_strengths = [0.1, 1, 10, 100]

    room_max_iter = 1500

    mlp_max_iter = 250

    ai_max_iter = 1000

photo_head_file = head_folder / 'photo_head.joblib'

aigen_head_file = head_folder / 'aigen_head.joblib'

sys.path.insert(0, str(project))

import numpy as np
import torch
import joblib
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
from transformers import CLIPModel, CLIPProcessor
from pretrained import PHOTO_LABELS as LABELS

SEED = 2108

RELEVANT = {'living_room', 'kitchen', 'bedroom', 'bathroom', 'dining_room', 'corridor'}

PHONE = ['a photo of a mobile phone', 'a photo of a smartphone', 'a photo of a telephone']

PROMPT_NAMES = [
    'living_room',
    'kitchen',
    'bedroom',
    'bathroom',
    'balcony',
    'service_yard',
    'window_view',
    'floor_plan',
    'empty_room',
    'corridor',
    'selfie',
    'person',
    'group_photo',
    'food',
    'dessert',
    'pet',
    'screenshot',
    'meme',
    'car',
    'outdoor',
]


def show_path(path):
    return path.relative_to(app_folder).as_posix()


print('Short training check' if short_run else 'Full training')

print('Iteration limits: room LogisticRegression', room_max_iter, '| room MLP', mlp_max_iter, '| AI LogisticRegression', ai_max_iter)

print('Room C values:', room_strengths)

print('Output folder for model files:', show_path(head_folder))

print('Output metrics file:', show_path(metrics_file))

print('Photo features are read from:', [show_path(file) for file in embedding_files])

print('New photo features would be written to:', show_path(embedding_file))

print('CLIP weights are only read from:', show_path(clip_folder))

/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Short training check
Iteration limits: room LogisticRegression 80 | room MLP 40 | AI LogisticRegression 80
Room C values: [0.1, 1]
Output folder for model files: training/check_runs/photo
Output metrics file: training/check_runs/photo/photo_metrics.json
Photo features are read from: ['training/photo_embeddings.npz', 'training/check_runs/photo/photo_embeddings.npz']
New photo features would be written to: training/check_runs/photo/photo_embeddings.npz
CLIP weights are only read from: flatfairproj/ml/clip


In [2]:
def find_cutoff(scores, rate):
    return float(np.nextafter(np.quantile(scores, 1 - rate, method='higher'), np.inf))


def room_stats(labels, scores, top_labels, block, review):
    real = np.isin(labels, list(RELEVANT))

    blocked = scores >= block

    catch_pass = {}

    for name in sorted(set(labels)):
        to_block = name not in RELEVANT

        catch_pass[name] = float(np.mean(blocked[labels == name] == to_block))

    top_recall = {}

    for name in sorted(set(labels)):
        top_recall[name] = float(np.mean(top_labels[labels == name] == name))

    return {
        'accuracy': float(np.mean(blocked == ~real)),
        'false_reject': float(blocked[real].mean()),
        'irrelevant_catch': float(blocked[~real].mean()),
        'review_false_flag': float((scores[real] >= review).mean()),
        'per_class_catch_or_accept': catch_pass,
        'top_label_recall': top_recall,
        'top_label_confusion': confusion_matrix(
            labels, top_labels, labels=sorted(set(labels) | set(top_labels))
        ).tolist(),
        'confusion_labels': sorted(set(labels) | set(top_labels)),
    }


def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def json_text(value):
    return json.dumps(value, sort_keys=True)


def show_report(report):
    print('Photo split:', report['split'])

    print('Phone photos dropped:', report['phone_dropped'])

    print('Room model validation accuracy:', report['candidates'])

    print('Chosen room model:', report['chosen'])

    print('Hybrid confidence:', report['hybrid_confidence'])

    print('Room thresholds:', report['thresholds'])

    print('Room test accuracy:', report['v2']['accuracy'])

    print('Room test false reject:', report['v2']['false_reject'])

    print('Room test irrelevant catch:', report['v2']['irrelevant_catch'])

    print('AI photo model uses frequency features:', report['ai']['frequency'])

    print('AI photo threshold:', report['ai']['threshold'])

    print('AI photo test AUC:', report['ai']['test_auc'])

    print('AI photo source macro AUC:', report['ai']['source_macro_auc'])

    print('Modern AI photos flagged:', report['ai']['modern_flagged'], 'of', report['ai']['modern_total'])

    for row in report['ai']['logo']:
        print('Left out generator:', row['generator'], '| AUC', row['auc'], '| recall', row['recall'])

In [3]:
saved_files = [
    saved_head_folder / 'photo_head.joblib',
    saved_head_folder / 'aigen_head.joblib',
    saved_metrics_file,
    saved_embedding_file,
]

saved_hashes = {file: file_hash(file) for file in saved_files if file.is_file()}

for file, saved_hash in saved_hashes.items():
    print('sha256', saved_hash, show_path(file))

if saved_metrics_file.is_file():
    saved_report = json.loads(saved_metrics_file.read_text())

    print('Saved results from an earlier run:', show_path(saved_metrics_file))

    show_report(saved_report)
else:
    print('No saved results from an earlier run:', show_path(saved_metrics_file), 'is missing')

sha256 59764c96b9de6e28eddbe3a44b964ab6c7ea5aad21f08ab4d732f3264cf07963 flatfairproj/ml/photo_head.joblib
sha256 b6c09c7c5c0c4a49efc9ef5b454c9307a2078fcbc1763d2d4d871b5a88243c8e flatfairproj/ml/aigen_head.joblib
sha256 899c4b0ac9faf81629523d92b828a17f01b2cad1ad9aad6b48593eadf864c552 training/photo_metrics.json
sha256 8681a155ab31383bd9d67f46acdb34be4f830595e8b0ce05515c79101b4797d1 training/photo_embeddings.npz
Saved results from an earlier run: training/photo_metrics.json
Photo split: {'train': 5392, 'val': 1155, 'test': 1156}
Phone photos dropped: 13
Room model validation accuracy: [[0.8805194805194805, '(0.1, None)'], [0.9238095238095239, "(0.1, 'balanced')"], [0.9393939393939394, '(1, None)'], [0.9454545454545454, "(1, 'balanced')"], [0.9489177489177489, '(10, None)'], [0.9515151515151515, "(10, 'balanced')"], [0.9471861471861471, '(100, None)'], [0.9471861471861471, "(100, 'balanced')"], [0.9532467532467532, 'MLP64']]
Chosen room model: MLP64
Hybrid confidence: 0.9
Room thresholds:

In [4]:
def find_pics():
    found = []

    for base in ['imgdata', 'synthdata']:
        for photo in (photo_folder / base).glob('*/*'):
            if photo.suffix.lower() in {'.jpg', '.png', '.jpeg'}:
                found.append(photo)

    return sorted(found)


def embed_words(clip, processor):
    prompts = [label[0] for label in LABELS] + PHONE

    with torch.inference_mode():
        tokens = processor(text=prompts, return_tensors='pt', padding=True)

        word_traits = clip.text_projection(clip.text_model(**tokens).pooler_output)

        word_traits = word_traits / word_traits.norm(dim=-1, keepdim=True)

    return word_traits


def wave_row(image):
    grey = np.asarray(image.resize((128, 128)).convert('L'), dtype=float) / 255

    power = np.abs(np.fft.fftshift(np.fft.fft2(grey - grey.mean()))) ** 2

    down, across = np.ogrid[-64:64, -64:64]

    radius = np.sqrt(across * across + down * down)

    return [power[radius > ring].sum() / (power.sum() + 1e-9) for ring in [16, 32, 48]]


def embed_pics(photos, clip, processor):
    names = [str(photo.relative_to(photo_folder)) for photo in photos]

    for file in embedding_files:
        if file.exists():
            saved = np.load(file)

            if saved['paths'].tolist() == names:
                print('Using saved photo features:', show_path(file), saved['X'].shape, saved['freq'].shape)

                return saved['X'], saved['freq']

            print('Saved photo features do not match the photos:', show_path(file), len(saved['paths']), 'saved,', len(names), 'found')

    pic_rows = []

    wave_rows = []

    for first in range(0, len(photos), 32):
        images = [Image.open(photo).convert('RGB') for photo in photos[first : first + 32]]

        with torch.inference_mode():
            pixels = processor(images=images, return_tensors='pt')['pixel_values']

            traits = clip.visual_projection(clip.vision_model(pixel_values=pixels).pooler_output)

            traits = traits / traits.norm(dim=-1, keepdim=True)

            pic_rows.extend(traits.numpy())

        for image in images:
            wave_rows.append(wave_row(image))

        print('embedded', min(first + 32, len(photos)), len(photos), flush=True)

    pic_traits = np.asarray(pic_rows)

    wave_traits = np.asarray(wave_rows)

    embedding_file.parent.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(embedding_file, X=pic_traits, freq=wave_traits, paths=names)

    print('Saved new photo features:', show_path(embedding_file))

    return pic_traits, wave_traits

In [5]:
torch.set_num_threads(4)

np.random.seed(SEED)

photos = find_pics()

print('Photos found:', len(photos))

clip = CLIPModel.from_pretrained(clip_folder, local_files_only=True).eval()

processor = CLIPProcessor.from_pretrained(clip_folder, local_files_only=True)

prompt_features = embed_words(clip, processor)

photo_features, frequency_features = embed_pics(photos, clip, processor)

print('Photo features:', photo_features.shape, '| frequency features:', frequency_features.shape)

Photos found: 9287


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 12599.79it/s]

Using saved photo features: training/photo_embeddings.npz (9287, 512) (9287, 3)
Photo features: (9287, 512) | frequency features: (9287, 3)


In [6]:
logit_scale = float(clip.logit_scale.exp().detach())

prompt_logits = photo_features @ prompt_features.numpy().T * logit_scale

room_prompt_logits = prompt_logits[:, :20]

prompt_scores = np.exp(room_prompt_logits - room_prompt_logits.max(1, keepdims=True))

prompt_scores /= prompt_scores.sum(1, keepdims=True)

photo_labels = np.array([photo.parent.name for photo in photos])

room_photo = np.array([photo.parent.parent.name == 'imgdata' for photo in photos])

phone = photo_labels == 'phone'

top_three_prompts = np.argsort(prompt_logits, axis=1)[:, -3:]

phone_like = np.any(top_three_prompts >= 20, axis=1)

phone_dropped = int((phone & ~phone_like).sum())

room_photo &= ~phone | phone_like

room_ids = np.where(room_photo)[0]

train_ids, rest_ids = train_test_split(
    room_ids, test_size=0.3, stratify=photo_labels[room_ids], random_state=SEED
)

val_ids, test_ids = train_test_split(
    rest_ids, test_size=0.5, stratify=photo_labels[rest_ids], random_state=SEED
)

In [7]:
def train_pics(photo_features, photo_labels, prompt_scores, train_ids, val_ids, test_ids, phone_dropped):
    pool = []

    for force in room_strengths:
        for weight in [None, 'balanced']:
            model = LogisticRegression(C=force, class_weight=weight, max_iter=room_max_iter, random_state=SEED)

            model = model.fit(photo_features[train_ids], photo_labels[train_ids])

            correct = float(model.score(photo_features[val_ids], photo_labels[val_ids]))

            pool.append((correct, str((force, weight)), model))

    model = MLPClassifier(hidden_layer_sizes=(64,), max_iter=mlp_max_iter, early_stopping=True, random_state=SEED)

    model = model.fit(photo_features[train_ids], photo_labels[train_ids])

    correct = float(model.score(photo_features[val_ids], photo_labels[val_ids]))

    pool.append((correct, 'MLP64', model))

    _, chosen, model = max(pool, key=lambda candidate: candidate[0])

    unfit = ~np.isin(model.classes_, list(RELEVANT))

    ai_scores = model.predict_proba(photo_features)[:, unfit].sum(1)

    fit_val = np.isin(photo_labels[val_ids], list(RELEVANT))

    best_text = prompt_scores.argmax(1)

    best_mark = prompt_scores.max(1)

    hybrid = []

    for trust in [0.7, 0.8, 0.9, 0.95]:
        trusted = np.isin(best_text, [4, 5, 6]) & (best_mark >= trust)

        pic_scores = np.where(trusted, 0, ai_scores)

        block = find_cutoff(pic_scores[val_ids][fit_val], 0.01)

        correct = float(np.mean((pic_scores[val_ids] >= block) == ~fit_val))

        hybrid.append((correct, trust, pic_scores, block))

    _, trust, pic_scores, block = max(hybrid, key=lambda attempt: attempt[0])

    review = find_cutoff(pic_scores[val_ids][fit_val], 0.03)

    base_marks = prompt_scores[:, 10:].sum(1)

    baseline_labels = np.array([PROMPT_NAMES[best] for best in best_text])

    top_tags = model.predict(photo_features).astype(object)

    trusted = np.isin(best_text, [4, 5, 6]) & (best_mark >= trust)

    top_tags[trusted] = np.asarray(PROMPT_NAMES)[best_text[trusted]]

    report = {
        'phone_dropped': phone_dropped,
        'split': {'train': len(train_ids), 'val': len(val_ids), 'test': len(test_ids)},
        'candidates': [(accuracy, name) for accuracy, name, _ in pool],
        'chosen': chosen,
        'hybrid_confidence': trust,
        'thresholds': {'block': block, 'review': review},
        'baseline': room_stats(photo_labels[test_ids], base_marks[test_ids], baseline_labels[test_ids], 0.8, 0.5),
        'v2': room_stats(photo_labels[test_ids], pic_scores[test_ids], top_tags[test_ids], block, review),
        'validation': room_stats(photo_labels[val_ids], pic_scores[val_ids], top_tags[val_ids], block, review),
    }

    return dict(
        model=model,
        labels=model.classes_.tolist(),
        relevant=sorted(RELEVANT),
        thresholds=report['thresholds'],
        hybrid_confidence=trust,
        metrics=report,
        version='photo-v2',
    )

In [8]:
if ai_only:
    report = json.loads(metrics_file.read_text())
else:
    photo_head = train_pics(
        photo_features, photo_labels, prompt_scores, train_ids, val_ids, test_ids, phone_dropped
    )

    report = photo_head['metrics']

/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


In [9]:
if not ai_only:
    photo_head_file.parent.mkdir(parents=True, exist_ok=True)

    joblib.dump(photo_head, photo_head_file)

    print('Saved', show_path(photo_head_file))

Saved training/check_runs/photo/photo_head.joblib


In [10]:
def train_ai(
    photos, photo_features, frequency_features, photo_labels, room_ids, train_ids, val_ids, test_ids
):
    with open(photo_folder / 'synthdata/manifest.csv') as stream:
        sheet_rows = list(csv.DictReader(stream))

    maker_names = {}

    for row in sheet_rows:
        maker_names[row['class'] + '/' + row['file']] = row['generator']

    generators = np.array([maker_names.get(photo.parent.name + '/' + photo.name, 'real') for photo in photos])

    is_ai = np.isin(photo_labels, ['ai_interior', 'ai_general'])

    is_old = np.isin(photo_labels, ['real_interior', 'real_general'])

    is_new = photo_labels == 'ai_modern_test'

    pic_pool = train_ids[~np.isin(photo_labels[train_ids], ['floor_plan', 'screenshot', 'meme'])]

    sample_ids, _ = train_test_split(
        pic_pool, train_size=int(is_old.sum()), stratify=photo_labels[pic_pool], random_state=SEED
    )

    real_ids = np.concatenate([np.where(is_old)[0], sample_ids])

    real_train, real_rest = train_test_split(
        real_ids, test_size=0.3, stratify=photo_labels[real_ids], random_state=SEED
    )

    real_val, real_test = train_test_split(
        real_rest, test_size=0.5, stratify=photo_labels[real_rest], random_state=SEED
    )

    mit_val = val_ids[np.isin(photo_labels[val_ids], list(RELEVANT))]

    mit_test = test_ids[np.isin(photo_labels[test_ids], list(RELEVANT))]

    laion_val = real_val[photo_labels[real_val] == 'real_interior']

    laion_test = real_test[photo_labels[real_test] == 'real_interior']

    tuning = np.r_[mit_val, laion_val]

    indoor_test = np.r_[mit_test, laion_test]

    assert not np.intersect1d(real_train, np.r_[tuning, indoor_test]).size

    assert not np.intersect1d(tuning, indoor_test).size

    ai_ids = np.where(is_ai)[0]

    ai_train, ai_rest = train_test_split(ai_ids, test_size=0.3, stratify=generators[ai_ids], random_state=SEED)

    ai_val, ai_test = train_test_split(ai_rest, test_size=0.5, stratify=generators[ai_rest], random_state=SEED)

    tries = []

    for wave in [False, True]:
        if wave:
            traits = np.concatenate([photo_features, frequency_features], axis=1)
        else:
            traits = photo_features

        fit_ids = np.r_[real_train, ai_train]

        model = LogisticRegression(C=1, class_weight='balanced', max_iter=ai_max_iter, random_state=SEED)

        model = model.fit(traits[fit_ids], is_ai[fit_ids])

        ai_threshold = find_cutoff(model.predict_proba(traits[tuning])[:, 1], 0.03)

        check_ids = np.r_[real_val, ai_val]

        auc = roc_auc_score(is_ai[check_ids], model.predict_proba(traits[check_ids])[:, 1])

        tries.append((auc, wave, traits, model, ai_threshold))

    _, wave, traits, model, ai_threshold = max(tries, key=lambda attempt: attempt[0])

    maker_rows = []

    for maker in sorted(set(generators[is_ai])):
        fit_ids = np.r_[real_train, np.where(is_ai & (generators != maker))[0]]

        rival = LogisticRegression(C=1, class_weight='balanced', max_iter=ai_max_iter, random_state=SEED)

        rival = rival.fit(traits[fit_ids], is_ai[fit_ids])

        calibration_scores = rival.predict_proba(traits[tuning])[:, 1]

        other_threshold = find_cutoff(calibration_scores, 0.03)

        check_ids = np.r_[real_test, np.where(is_ai & (generators == maker))[0]]

        check_scores = rival.predict_proba(traits[check_ids])[:, 1]

        interior_test_scores = rival.predict_proba(traits[indoor_test])[:, 1]

        maker_rows.append(
            {
                'generator': maker,
                'auc': float(roc_auc_score(is_ai[check_ids], check_scores)),
                'recall': float((check_scores[is_ai[check_ids]] >= other_threshold).mean()),
                'threshold': other_threshold,
                'real_interior_val_false_flag': float((calibration_scores >= other_threshold).mean()),
                'real_interior_test_false_flag': float((interior_test_scores >= other_threshold).mean()),
            }
        )

    source_auc = {}

    for name in sorted(set(photo_labels[real_test])):
        check_ids = np.r_[real_test[photo_labels[real_test] == name], ai_test]

        source_auc[name] = float(roc_auc_score(is_ai[check_ids], model.predict_proba(traits[check_ids])[:, 1]))

    origins = {
        'LAION originals': real_test[np.isin(photo_labels[real_test], ['real_general', 'real_interior'])],
        'imgdata photos': real_test[~np.isin(photo_labels[real_test], ['real_general', 'real_interior'])],
    }

    even = {}

    for name, group_ids in origins.items():
        check_ids = np.r_[group_ids, ai_test]

        even[name] = {
            'real_count': len(group_ids),
            'auc': float(roc_auc_score(is_ai[check_ids], model.predict_proba(traits[check_ids])[:, 1])),
            'false_flag': float((model.predict_proba(traits[group_ids])[:, 1] >= ai_threshold).mean()),
        }

    check_ids = np.r_[real_test, ai_test]

    ai_report = {
        'source_balanced': even,
        'source_macro_auc': float(np.mean([group['auc'] for group in even.values()])),
        'source_auc': source_auc,
        'feature_attempts': [(auc, frequency) for auc, frequency, *_ in tries],
        'frequency': wave,
        'logo': maker_rows,
        'threshold': ai_threshold,
        'modern_flagged': int((model.predict_proba(traits[is_new])[:, 1] >= ai_threshold).sum()),
        'modern_total': int(is_new.sum()),
        'real_calibration_interior_count': len(tuning),
        'source_false_flags': {
            name: float(
                (model.predict_proba(traits[real_test][photo_labels[real_test] == name])[:, 1] >= ai_threshold).mean()
            )
            for name in sorted(set(photo_labels[real_test]))
        },
        'test_auc': float(roc_auc_score(is_ai[check_ids], model.predict_proba(traits[check_ids])[:, 1])),
    }

    fit_ids = np.r_[real_train, ai_ids]

    model.fit(traits[fit_ids], is_ai[fit_ids])

    ai_threshold = find_cutoff(model.predict_proba(traits[tuning])[:, 1], 0.03)

    ai_report['threshold'] = ai_threshold

    ai_report['modern_flagged'] = int((model.predict_proba(traits[is_new])[:, 1] >= ai_threshold).sum())

    ai_report['modern_probabilities'] = model.predict_proba(traits[is_new])[:, 1].tolist()

    ai_report['final_source_false_flags'] = {
        name: float(
            (model.predict_proba(traits[real_test][photo_labels[real_test] == name])[:, 1] >= ai_threshold).mean()
        )
        for name in sorted(set(photo_labels[real_test]))
    }

    def flag_counts(ids):
        scores = model.predict_proba(traits[ids])[:, 1]

        return {
            'count': len(ids),
            'flagged': int((scores >= ai_threshold).sum()),
            'false_flag': float((scores >= ai_threshold).mean()),
        }

    ai_report['interior_calibration'] = {
        'VAL': flag_counts(tuning),
        'TEST': flag_counts(indoor_test),
        'VAL_MIT': flag_counts(mit_val),
        'TEST_MIT': flag_counts(mit_test),
        'VAL_LAION': flag_counts(laion_val),
        'TEST_LAION': flag_counts(laion_test),
    }

    new_ids = np.where(is_new)[0]

    new_checks = np.r_[indoor_test, new_ids]

    new_truth = np.r_[np.zeros(len(indoor_test)), np.ones(len(new_ids))]

    ai_report['modern_vs_real_interior_test_auc'] = float(
        roc_auc_score(new_truth, model.predict_proba(traits[new_checks])[:, 1])
    )

    old_pool = room_ids[~np.isin(photo_labels[room_ids], ['floor_plan', 'screenshot', 'meme'])]

    old_sample, _ = train_test_split(
        old_pool, train_size=int(is_old.sum()), stratify=photo_labels[old_pool], random_state=SEED
    )

    old_real = np.r_[np.where(is_old)[0], old_sample]

    old_train, _ = train_test_split(old_real, test_size=0.3, stratify=photo_labels[old_real], random_state=SEED)

    ai_report['prior_real_training_overlap'] = {
        'photo_VAL_MIT': len(np.intersect1d(old_train, mit_val)),
        'photo_TEST_MIT': len(np.intersect1d(old_train, mit_test)),
        'new_overlap': 0,
    }

    ai_report['calibration_method'] = (
        '97th percentile on disjoint photo-VAL MIT interiors plus held-out LAION interiors; '
        'modern images evaluation only'
    )

    ai_report['split_paths'] = {}

    for name, split_ids in [
        ('real_train', real_train),
        ('interior_val', tuning),
        ('interior_test', indoor_test),
    ]:
        ai_report['split_paths'][name] = [str(photos[i].relative_to(photo_folder)) for i in split_ids]

    for row in maker_rows:
        maker = row['generator']

        maker_ids = np.where(is_ai & (generators == maker))[0]

        fit_ids = np.r_[real_train, np.where(is_ai & (generators != maker))[0]]

        rival = LogisticRegression(C=1, class_weight='balanced', max_iter=ai_max_iter, random_state=SEED)

        rival = rival.fit(traits[fit_ids], is_ai[fit_ids])

        generator_scores = rival.predict_proba(traits[maker_ids])[:, 1]

        interior_test_scores = rival.predict_proba(traits[indoor_test])[:, 1]

        row['recall_at_final_threshold'] = float((generator_scores >= ai_threshold).mean())

        row['interior_test_false_flag_at_final_threshold'] = float((interior_test_scores >= ai_threshold).mean())

    return dict(
        model=model,
        labels=['real', 'ai'],
        relevant=sorted(RELEVANT),
        frequency=wave,
        thresholds={'review': ai_threshold},
        metrics=ai_report,
        version='photo-v2',
    )

In [11]:
aigen_head = train_ai(
    photos, photo_features, frequency_features, photo_labels, room_ids, train_ids, val_ids, test_ids
)

In [12]:
aigen_head_file.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(aigen_head, aigen_head_file)

report = {**report, 'ai': aigen_head['metrics']}

metrics_file.parent.mkdir(parents=True, exist_ok=True)

metrics_file.write_text(json.dumps(report, indent=2))

print('Saved', show_path(aigen_head_file))

print('Saved', show_path(metrics_file))

if short_run:
    print('Results from this short training check')

    show_report(report)
else:
    print(json.dumps({key: value for key, value in report.items() if key not in ['baseline', 'v2']}, indent=2), flush=True)

Saved training/check_runs/photo/aigen_head.joblib
Saved training/check_runs/photo/photo_metrics.json
Results from this short training check
Photo split: {'train': 5392, 'val': 1155, 'test': 1156}
Phone photos dropped: 13
Room model validation accuracy: [(0.8805194805194805, '(0.1, None)'), (0.9238095238095239, "(0.1, 'balanced')"), (0.9393939393939394, '(1, None)'), (0.9454545454545454, "(1, 'balanced')"), (0.9532467532467532, 'MLP64')]
Chosen room model: MLP64
Hybrid confidence: 0.9
Room thresholds: {'block': 0.5137496590614319, 'review': 0.3526867926120758}
Room test accuracy: 0.986159169550173
Room test false reject: 0.018518518518518517
Room test irrelevant catch: 0.9879807692307693
AI photo model uses frequency features: False
AI photo threshold: 0.49403029680252075
AI photo test AUC: 0.9502816901408451
AI photo source macro AUC: 0.9497797619047619
Modern AI photos flagged: 6 of 8
Left out generator: CompVis/stable-diffusion-v1-4 | AUC 0.9368742575937553 | recall 0.801204819277108

In [13]:
room_report = {key: value for key, value in report.items() if key != 'ai'}

loaded_report = json.loads(metrics_file.read_text())

assert json_text(loaded_report) == json_text(report)

print('Metrics file equals the report from this run:', show_path(metrics_file))

if not ai_only:
    loaded_photo_head = joblib.load(photo_head_file)

    assert json_text(loaded_photo_head['metrics']) == json_text(room_report)

    assert loaded_photo_head['thresholds'] == report['thresholds']

    print('photo_head metrics equal the report from this run without ai:', show_path(photo_head_file))

    print('photo_head model:', type(loaded_photo_head['model']).__name__, '| iterations', np.max(loaded_photo_head['model'].n_iter_))

loaded_aigen_head = joblib.load(aigen_head_file)

assert json_text(loaded_aigen_head['metrics']) == json_text(report['ai'])

assert loaded_aigen_head['thresholds']['review'] == report['ai']['threshold']

print('aigen_head metrics equal the ai part of the report from this run:', show_path(aigen_head_file))

print('aigen_head model:', type(loaded_aigen_head['model']).__name__, '| iterations', np.max(loaded_aigen_head['model'].n_iter_))

for file in [photo_head_file, aigen_head_file, metrics_file]:
    if file.is_file():
        print('sha256', file_hash(file), show_path(file))

if short_run:
    for file, saved_hash in saved_hashes.items():
        assert file_hash(file) == saved_hash, show_path(file)

    print('Earlier saved files unchanged:', [show_path(file) for file in saved_hashes])

Metrics file equals the report from this run: training/check_runs/photo/photo_metrics.json
photo_head metrics equal the report from this run without ai: training/check_runs/photo/photo_head.joblib
photo_head model: MLPClassifier | iterations 40
aigen_head metrics equal the ai part of the report from this run: training/check_runs/photo/aigen_head.joblib
aigen_head model: LogisticRegression | iterations 16
sha256 41ae95817f0385e2604fbe51053183aaaa7098162f86d12e0856ac1dc81780a8 training/check_runs/photo/photo_head.joblib
sha256 6accb736820a66da9e8cba2db871c8a01e0301954f95ad6844525c1c2ae365d2 training/check_runs/photo/aigen_head.joblib
sha256 5c50eb52fec20bfe0e75ecb2cbc8f66c5d27cba673f083ed5c8d9c77421b4e0c training/check_runs/photo/photo_metrics.json
Earlier saved files unchanged: ['flatfairproj/ml/photo_head.joblib', 'flatfairproj/ml/aigen_head.joblib', 'training/photo_metrics.json', 'training/photo_embeddings.npz']
